# Part 4 — Application: Multimodal Care AI Voice Call

**Models:** MedGemma-4B (nurse LLM) + MedGemma-Vision 1.5-4B (visual AE) + HeAR (cough detection)  
**Task:** End-to-end multimodal pipeline — camera/mic → SAE detection → nurse conversation → clinical assessment  
**Evaluation:** 6 prompt strategies × multimodal ablation (visual + audio + text contributions)

---

This notebook is part of the **MedGemma Clinical Trial Engine** pipeline:

```
NB1  Anti-Hallucination ──── RLFR fine-tuning for reliable medical text (MedGemma)
NB2  SAE Detection ────────── MedGemma 1.5 + MedSigLIP + HeAR (image + audio → AE)
NB3  Clinical Trial Sim ──── Rule set generation + hazard-based daily simulation
NB4  Voice Call App ────────── MedGemma 4B virtual nurse (multi-turn dialogue)
NB5  SAE Report Gen ────────── CRF data → MedWatch 3500A pharmacovigilance reports
```

In [ ]:
# ── Install dependencies ──
!pip install -q torch transformers accelerate Pillow matplotlib numpy huggingface-hub pandas
!pip install -q tensorflow librosa soundfile joblib scikit-learn

# ── Install this project (for src.* imports) ──
!pip install -q -e ..

# ── HuggingFace login ──
# Required for gated models:
#   - https://huggingface.co/google/medgemma-1.5-4b-it  (visual detection)
#   - https://huggingface.co/google/medgemma-4b-it       (nurse LLM)
#   - https://huggingface.co/google/hear                 (cough detection)
import os
from huggingface_hub import login

HF_TOKEN = ""  # <-- paste your HuggingFace token here

if HF_TOKEN:
    login(token=HF_TOKEN)
elif os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
else:
    login()

In [ ]:
import sys, os, json, re, time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import Counter

from transformers import (
    AutoModelForImageTextToText, AutoProcessor,
    AutoModel, AutoImageProcessor,
    AutoTokenizer, AutoModelForCausalLM,
)
from huggingface_hub import snapshot_download
import pandas as pd
from IPython.display import display, HTML, Markdown, Audio

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

DATA_DIR = Path(snapshot_download("AlphaRaven/clinical-trial-engine-data", repo_type="dataset"))

IMAGE_DIR       = DATA_DIR / "ae_images"
AUDIO_DIR       = DATA_DIR / "cough_audio_samples"
SIGLIP_FT_DIR   = DATA_DIR / "siglip_ft_head"
GPU_ID = 0   # ← change to your GPU index

from src.multimodal_v2.config import build_ctcae_table_text, get_config, SIGLIP_CLASSES
cfg = get_config()
CLASSES = SIGLIP_CLASSES

print(f"Dataset:  {DATA_DIR}")
print(f"Images:   {IMAGE_DIR}")
print(f"Audio:    {AUDIO_DIR}")
print(f"GPU:      cuda:{GPU_ID}")

---

## Part 1 — Visual AE Detection

Using the **finetuned MedGemma-Vision** model (from NB2) to detect adverse events
by comparing a baseline photograph (pre-treatment) with the current image (during treatment).

In [ ]:
device = torch.device(f"cuda:{GPU_ID}")
GEMMA_FT_MODEL_ID = "AlphaRaven/medgemma-ae-detection"

print("Loading MedGemma-Vision finetuned ...")
gemma_processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it")
gemma_vision = AutoModelForImageTextToText.from_pretrained(
    GEMMA_FT_MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": GPU_ID},
)
gemma_vision.eval()
print(f"  Params: {sum(p.numel() for p in gemma_vision.parameters())/1e9:.2f}B")

# ── Load test patient from NB2 dataset ──
metadata = json.loads((IMAGE_DIR / "image_metadata.json").read_text())
baseline_map = {}
for e in metadata["images"]:
    if e["class_name"] == "normal":
        key = f"{e['profile']['age']}/{e['profile']['sex']}/{e['profile']['race']}"
        baseline_map[key] = e["file"]

test_entries = [e for e in metadata["images"] if e["split"] == "test" and e["class_name"] != "normal"]
patient_img = test_entries[0]
pkey = f"{patient_img['profile']['age']}/{patient_img['profile']['sex']}/{patient_img['profile']['race']}"
baseline_file = baseline_map.get(pkey)

print(f"\nPatient profile: {patient_img['profile']}")
print(f"Ground truth:    {patient_img['class_name']}")
print(f"Baseline image:  {baseline_file}")
print(f"Current image:   {patient_img['file']}")

In [ ]:
# ── Run visual AE detection ──
CTCAE_TABLE = build_ctcae_table_text()
PROMPT = f"""You are a dermatology AI analyzing a patient during a clinical trial.

CTCAE reference:
{CTCAE_TABLE}

Image 1 = BASELINE (pre-treatment). Image 2 = CURRENT (during treatment).
Compare and detect any new adverse events.

Return a JSON array of detected AEs. Each element:
{{{{
  "ae_term": "<category>",
  "grade": <1|2|3>,
  "reasoning": "<clinical description of what changed from Image 1 to Image 2>"
}}}}

If no change from baseline is detected, return: []"""

baseline_path = IMAGE_DIR / baseline_file
current_path = IMAGE_DIR / patient_img["file"]
baseline_img = Image.open(baseline_path).convert("RGB")
current_img = Image.open(current_path).convert("RGB")

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": baseline_img},
        {"type": "image", "image": current_img},
        {"type": "text", "text": PROMPT},
    ],
}]

inputs = gemma_processor.apply_chat_template(
    messages, add_generation_prompt=True,
    tokenize=True, return_dict=True, return_tensors="pt",
).to(device)
inputs.pop("token_type_ids", None)
input_len = inputs["input_ids"].shape[-1]

t0 = time.time()
with torch.inference_mode():
    output = gemma_vision.generate(**inputs, max_new_tokens=1024, do_sample=False)
vis_latency = time.time() - t0

raw_text = gemma_processor.decode(output[0][input_len:], skip_special_tokens=True).strip()

detected_aes = []
try:
    detected_aes = json.loads(raw_text)
except json.JSONDecodeError:
    match = re.search(r'\[.*?\]', raw_text, re.DOTALL)
    if match:
        try:
            detected_aes = json.loads(match.group())
        except json.JSONDecodeError:
            pass

# Convert to Care AI visual_assessment format
GRADE_SEVERITY = {1: "mild", 2: "moderate", 3: "severe"}
visual_assessment = {
    "source": "MedGemma-Vision (finetuned)",
    "findings": [
        {
            "ae_term": ae["ae_term"],
            "grade": ae["grade"],
            "observation": f"{ae['ae_term'].replace('_', ' ')} detected",
            "visual_evidence": ae.get("reasoning", ""),
            "estimated_severity": GRADE_SEVERITY.get(ae["grade"], "moderate"),
            "confidence": 0.85,
            "reasoning": ae.get("reasoning", ""),
        }
        for ae in detected_aes
    ],
    "general_observations": [],
}

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(baseline_img); axes[0].set_title("Baseline (pre-treatment)"); axes[0].axis("off")
axes[1].imshow(current_img); axes[1].set_title("Current (during treatment)"); axes[1].axis("off")
plt.suptitle(
    f"Patient: {patient_img['profile']['age']}y {patient_img['profile']['sex']}  |  GT: {patient_img['class_name']}",
    fontsize=12,
)
plt.tight_layout()
plt.show()

print(f"\n── Visual Detection Result ({vis_latency:.1f}s) ──")
if detected_aes:
    for ae in detected_aes:
        print(f"  {ae['ae_term']} Grade {ae['grade']}: {ae.get('reasoning', '')[:120]}")
else:
    print("  No AEs detected")
print(f"\nGround truth: {patient_img['class_name']}")

In [ ]:
del gemma_vision, gemma_processor
torch.cuda.empty_cache()
print("MedGemma-Vision unloaded — GPU memory freed for next model.")

---

## Part 2 — Cough Detection

Using Google's **HeAR** (Health Acoustic Representations) model with a 2-stage classifier:
- **Stage 1**: Is there a cough? (none / dry / wet)
- **Stage 2**: If cough detected, refine: dry or wet?

Clinical significance:
- **Dry cough** → pneumonitis, interstitial lung disease
- **Wet cough** → infection, fluid overload, pulmonary edema

In [ ]:
# ── Load HeAR on CPU (keep GPU free for PyTorch) ──
HEAR_AVAILABLE = False
audio_assessment = ""
predicted_cough = "none"

try:
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    import tensorflow as tf
    tf.config.set_visible_devices([], 'GPU')
    import librosa
    from joblib import load as joblib_load
    from src.cough_detection.segmentation import segment_cough

    SR = 16000

    hear_path = snapshot_download("google/hear", repo_type="model")
    hear_model = tf.saved_model.load(hear_path)
    hear_serving = hear_model.signatures["serving_default"]

    clf_3c = joblib_load(str(DATA_DIR / "hear_mixed_model" / "classifier.joblib"))
    le_3c  = joblib_load(str(DATA_DIR / "hear_mixed_model" / "label_encoder.joblib"))
    clf_2c = joblib_load(str(DATA_DIR / "hear_cough_only_model" / "classifier.joblib"))
    le_2c  = joblib_load(str(DATA_DIR / "hear_cough_only_model" / "label_encoder.joblib"))

    HEAR_AVAILABLE = True
    print("HeAR model loaded (CPU).")

except Exception as e:
    print(f"HeAR not available ({e}). Using pre-computed result.")
    audio_assessment = (
        "Dry cough detected (4/6 segments classified as cough). "
        "Possible pneumonitis indicator — recommend pulmonary function assessment."
    )
    predicted_cough = "dry"
    print(f"Fallback audio_assessment: {audio_assessment}")

In [ ]:
if HEAR_AVAILABLE:
    def get_embedding(audio_seg):
        target_len = 2 * SR
        if len(audio_seg) < target_len:
            audio_seg = np.pad(audio_seg, (0, target_len - len(audio_seg)))
        else:
            audio_seg = audio_seg[:target_len]
        waveform = tf.constant(audio_seg[np.newaxis, :], dtype=tf.float32)
        result = hear_serving(waveform)
        emb_key = [k for k in result.keys() if "embedding" in k.lower()][0]
        return result[emb_key].numpy()

    # Select a cough audio sample
    audio_files = sorted((AUDIO_DIR / "dry").glob("*.wav"))
    if not audio_files:
        audio_files = sorted((AUDIO_DIR / "wet").glob("*.wav"))
    audio_path = audio_files[0]
    audio_gt = audio_path.parent.name

    audio_raw, sr = librosa.load(str(audio_path), sr=SR)
    duration = len(audio_raw) / sr

    print(f"Audio file: {audio_path.name} ({duration:.1f}s)")
    print(f"Ground truth: {audio_gt} cough")
    display(Audio(audio_raw, rate=SR))

    # ── Run 2-stage cough detection pipeline ──
    segs, msk = segment_cough(audio_raw, sr, cough_padding=0)
    ch = np.diff(msk.astype(int))
    ss = np.where(ch == 1)[0] + 1
    ee = np.where(ch == -1)[0] + 1

    seg_results = []
    for j, (s, e) in enumerate(zip(ss, ee)):
        if j >= len(segs):
            break
        emb = get_embedding(segs[j])

        pred_3c = clf_3c.predict(emb)[0]
        proba_3c = clf_3c.predict_proba(emb)[0]
        label_3c = le_3c.inverse_transform([pred_3c])[0]

        if label_3c == 'none':
            final_label = 'none'
        else:
            pred_2c = clf_2c.predict(emb)[0]
            label_2c = le_2c.inverse_transform([pred_2c])[0]
            final_label = label_2c

        seg_results.append({"start": round(s / sr, 3), "end": round(e / sr, 3), "label": final_label})

    cough_labels = [r["label"] for r in seg_results if r["label"] in ("dry", "wet")]
    predicted_cough = Counter(cough_labels).most_common(1)[0][0] if cough_labels else "none"
    n_cough = len(cough_labels)
    n_total = len(seg_results)

    if predicted_cough != "none":
        cough_implication = (
            "Dry cough may indicate pneumonitis or interstitial lung disease."
            if predicted_cough == "dry" else
            "Wet cough may indicate infection or pulmonary edema."
        )
        audio_assessment = (
            f"{predicted_cough.capitalize()} cough detected "
            f"({n_cough}/{n_total} segments classified as cough). "
            f"{cough_implication} Recommend pulmonary function assessment."
        )
    else:
        audio_assessment = "No cough detected in audio analysis."

    # Visualize
    fig, ax = plt.subplots(figsize=(10, 2.5))
    t = np.arange(len(audio_raw)) / sr
    ax.plot(t, audio_raw, color="gray", alpha=0.5, linewidth=0.5)
    colors = {"dry": "#e74c3c", "wet": "#3498db", "none": "#95a5a6"}
    for r in seg_results:
        ax.axvspan(r["start"], r["end"], alpha=0.3, color=colors.get(r["label"], "gray"))
    patches = [
        plt.Rectangle((0, 0), 1, 1, color=colors[l], alpha=0.3)
        for l in ["dry", "wet", "none"] if any(r["label"] == l for r in seg_results)
    ]
    labels = [l for l in ["dry", "wet", "none"] if any(r["label"] == l for r in seg_results)]
    ax.legend(patches, labels, loc="upper right")
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude")
    ax.set_title(f"Cough Detection — GT: {audio_gt} | Predicted: {predicted_cough} ({n_cough}/{n_total} cough segments)")
    plt.tight_layout()
    plt.show()

    print(f"\n── Audio Assessment ──")
    print(f"  {audio_assessment}")
    print(f"  GT: {audio_gt} | Predicted: {predicted_cough} | Match: {'O' if predicted_cough == audio_gt else 'X'}")

else:
    print("── Using pre-computed audio assessment (HeAR not loaded) ──")
    print(f"  {audio_assessment}")

---

## Part 3 — Build Patient Scenario

We construct a clinical trial patient scenario that integrates **real** multimodal detection results:

| Input | Source | Content |
|-------|--------|---------|
| Visual assessment | MedGemma-Vision (Part 1) | AE findings from patient photograph |
| Audio assessment | HeAR (Part 2) | Cough type classification |
| Non-visual AEs | Mock (hidden from nurse) | Must be detected through conversation |
| Patient persona | Simulated | Stoic minimizer — tends to downplay symptoms |

In [ ]:
from src.engine.mood import MoodState, compute_interaction_quality

# ── Parse visual GT ──
gt_class = patient_img["class_name"]
gt_parts = gt_class.rsplit("_", 1)
if gt_parts[-1].startswith("g") or gt_parts[-1].startswith("grade"):
    gt_ae_term = "_".join(gt_parts[:-1])
    gt_ae_grade = int(re.search(r'\d+', gt_parts[-1]).group())
else:
    gt_ae_term = gt_class
    gt_ae_grade = 1

DRUG_NAME = "Paclitaxel + Carboplatin"
INDICATION = "Non-small cell lung cancer (NSCLC)"
TREATMENT_DAY = 28

# ── Ground truth AEs ──
gt_visual_aes = [
    {"ae_term": gt_ae_term.replace("_", " "), "grade": gt_ae_grade,
     "video_signs": ["visible skin changes", "erythema"]},
]

gt_cough_ae = []
if predicted_cough != "none":
    gt_cough_ae = [{"ae_term": "cough", "grade": 1, "symptom_description": f"{predicted_cough} cough"}]

gt_non_visual_aes = [
    {"ae_term": "fatigue", "grade": 1, "symptom_description": "feeling tired and low energy"},
    {"ae_term": "nausea", "grade": 1, "symptom_description": "mild stomach discomfort, reduced appetite"},
    {"ae_term": "peripheral_neuropathy", "grade": 1, "symptom_description": "tingling in fingertips"},
] + gt_cough_ae

all_gt_aes = gt_visual_aes + gt_non_visual_aes

# ── Patient persona ──
mood = MoodState(persona_type="stoic_minimizer", seed=42)
quality = compute_interaction_quality(mood)

# ── Drug AE profile (common AEs for this regimen) ──
drug_ae_profile = [
    {"ae_term": "neutropenia", "incidence_pct": "80%", "common_symptoms": "increased infection risk, fever"},
    {"ae_term": "anaemia", "incidence_pct": "70%", "common_symptoms": "fatigue, weakness, pallor"},
    {"ae_term": "nausea", "incidence_pct": "65%", "common_symptoms": "stomach discomfort, reduced appetite"},
    {"ae_term": "peripheral_neuropathy", "incidence_pct": "60%", "common_symptoms": "tingling, numbness in hands/feet"},
    {"ae_term": "alopecia", "incidence_pct": "55%", "common_symptoms": "hair thinning or loss"},
    {"ae_term": "fatigue", "incidence_pct": "50%", "common_symptoms": "persistent tiredness, low energy"},
    {"ae_term": "thrombocytopenia", "incidence_pct": "40%", "common_symptoms": "easy bruising, bleeding"},
    {"ae_term": "rash", "incidence_pct": "15%", "common_symptoms": "skin redness, itching, papules"},
    {"ae_term": "pneumonitis", "incidence_pct": "5%", "common_symptoms": "cough, shortness of breath"},
    {"ae_term": "cough", "incidence_pct": "20%", "common_symptoms": "dry or productive cough"},
]

# ── Complete scenario ──
scenario = {
    "drug_name": DRUG_NAME,
    "indication": INDICATION,
    "visual_assessment": visual_assessment,
    "audio_assessment": audio_assessment,
    "drug_ae_profile": drug_ae_profile,
    "treatment_day": TREATMENT_DAY,
    "gt_non_visual_aes": gt_non_visual_aes,
    "gt_visual_aes": gt_visual_aes,
    "patient_demographics": {
        "age": patient_img["profile"]["age"],
        "sex": patient_img["profile"]["sex"],
    },
    "patient_persona_type": "stoic_minimizer",
    "patient_mood": mood.to_dict(),
}

# ── Patient's initial greeting (T1) ──
t1_visible = {
    "greeting": "Hi there. I'm managing, you know. Same old.",
    "reported_symptoms": [
        {"symptom": "I've had this skin thing going on, a bit red and bumpy. Not too bad."},
        {"symptom": "Been coughing here and there. Probably nothing."},
    ],
    "omitted_symptoms": ["fatigue", "nausea", "tingling in fingers"],
}

print("── Patient Scenario ──")
print(f"Drug:    {DRUG_NAME} for {INDICATION}")
print(f"Day:     {TREATMENT_DAY}")
print(f"Patient: {patient_img['profile']['age']}y {patient_img['profile']['sex']}, stoic minimizer")
print(f"\nGround-truth AEs (hidden from nurse):")
for ae in all_gt_aes:
    src = "visual" if ae in gt_visual_aes else "audio" if ae in gt_cough_ae else "non-visual"
    print(f"  - {ae['ae_term']} (Grade {ae['grade']}) [{src}]")
print(f"\nMultimodal inputs available to nurse:")
print(f"  Camera:  {len(visual_assessment['findings'])} finding(s) from MedGemma-Vision")
print(f"  Mic:     {audio_assessment[:80]}")
print(f"  Speech:  Patient greeting + {len(t1_visible['reported_symptoms'])} reported symptoms")
print(f"  Hidden:  {len(t1_visible['omitted_symptoms'])} symptoms the patient is NOT reporting")

---

## Part 4 — Nurse Conversation: Prompt Strategy Comparison

We test 6 system prompt designs that vary in context provided to MedGemma-4B:

| Strategy | What the model receives | Key question |
|---|---|---|
| **Baseline** | Drug name, visual assessment, AE profile | Does a standard prompt work? |
| **Concise** | Same + output length constraint | Does brevity improve focus? |
| **Full Context** | + patient personality, mood, interaction tips | Does knowing the patient help? |
| **No Drug Info** | Only visual assessment (no AE profile) | Can the model use its own knowledge? |
| **Few-Shot** | Full Context + example Q&A per AE | Do examples improve targeting? |
| **Deployment-Ready** | Drug + visual + **audio** + speaking style | **What's realistic in production?** |

Note: Only **Deployment-Ready** receives both visual AND audio assessment — matching the real deployment scenario.

In [ ]:
MODEL_PATH = "google/medgemma-4b-it"

print("Loading MedGemma-4B ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
nurse_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, device_map={"": device},
)
nurse_model.eval()

def nurse_fn(system_prompt: str, user_prompt: str) -> dict:
    chat = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(nurse_model.device)
    with torch.inference_mode():
        out = nurse_model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, do_sample=True)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    try:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(match.group()) if match else {"_raw": raw[:800]}
    except (json.JSONDecodeError, AttributeError):
        return {"_raw": raw[:800]}

print(f"MedGemma-4B loaded on GPU {GPU_ID}")

In [ ]:
from src.experiments.eval_prompt_templates import (
    template_a_current, template_b_concise, template_c_rich_persona,
    template_d_minimal, template_e_fewshot, template_f_realistic,
)

STRATEGIES = {
    "Baseline":         template_a_current,
    "Concise":          template_b_concise,
    "Full Context":     template_c_rich_persona,
    "No Drug Info":     template_d_minimal,
    "Few-Shot":         template_e_fewshot,
    "Deployment-Ready": template_f_realistic,
}

STRATEGY_DESC = {
    "Baseline":         "Drug + visual + AE profile (control)",
    "Concise":          "Same context, constrained output length",
    "Full Context":     "+ patient personality, mood, interaction tips",
    "No Drug Info":     "Visual assessment only — no AE profile",
    "Few-Shot":         "Full context + example questions per AE",
    "Deployment-Ready": "Drug + visual + AUDIO + speaking style",
}

history = [{**t1_visible, "_role": "patient", "_turn": 1}]

results = []
for strategy_name, tmpl_fn in STRATEGIES.items():
    sys_prompt, usr_prompt = tmpl_fn(scenario, quality, turn=2, history=history)
    t0 = time.time()
    response = nurse_fn(sys_prompt, usr_prompt)
    elapsed = time.time() - t0

    ack = response.get("acknowledgment") or response.get("_raw", "")[:200] or ""
    questions = response.get("questions", [])
    targets = [str(q.get("target_ae", "") or "") for q in questions if isinstance(q, dict)]
    style = response.get("approach_style") or "?"
    concerns = response.get("preliminary_concerns") or response.get("concerns") or []

    results.append({
        "strategy": strategy_name,
        "approach_style": style,
        "acknowledgment": ack[:200] if isinstance(ack, str) else str(ack)[:200],
        "n_questions": len(questions),
        "target_aes": ", ".join(targets),
        "concerns": concerns if isinstance(concerns, list) else [],
        "time_s": round(elapsed, 1),
        "_full_response": response,
    })
    print(f"  {strategy_name:18s}  {len(questions)} questions  targets=[{', '.join(targets)[:60]}]  {elapsed:.1f}s")

print(f"\nAll {len(results)} strategies completed.")

In [ ]:
def normalize_ae_term(term: str) -> str:
    return term.strip().lower().replace(" ", "_").replace("-", "_")

gt_terms = {normalize_ae_term(ae["ae_term"]) for ae in all_gt_aes}

def check_hit(target_str):
    targets = {normalize_ae_term(t.strip()) for t in target_str.split(",") if t.strip()}
    hits = sum(1 for g in gt_terms if any(g in t or t in g for t in targets))
    return f"{hits}/{len(gt_terms)}"

df = pd.DataFrame([{
    "Prompt Strategy": r["strategy"],
    "Tone": r["approach_style"],
    "Questions": r["n_questions"],
    "AEs Targeted": r["target_aes"][:80],
    "GT Hit": check_hit(r["target_aes"]),
    "Latency": f"{r['time_s']}s",
} for r in results])

display(HTML("<h3>Nurse Response — AE Targeting Accuracy (Round 1)</h3>"))
display(HTML(f"<p>Ground truth AEs: <b>{', '.join(gt_terms)}</b></p>"))
display(df.style.set_properties(**{"text-align": "left"}))

In [ ]:
for r in results:
    print(f"\n{'='*70}")
    print(f"  {r['strategy']} — {STRATEGY_DESC[r['strategy']]}")
    print(f"{'='*70}")
    print(f"  Tone: {r['approach_style']}")
    print(f"  Acknowledgment: {r['acknowledgment'][:250]}")
    print(f"  Questions:")
    for q in r["_full_response"].get("questions", []):
        if isinstance(q, dict):
            print(f"    → [{q.get('target_ae','?')}] {str(q.get('question',''))[:120]}")
    if r['concerns']:
        print(f"  Clinical concerns: {r['concerns']}")

---

## Part 5 — Clinical Assessment & Scoring

After the conversation, the nurse generates a **final clinical assessment**:
- Suspected AEs with estimated CTCAE grade and evidence
- Recommended action: `no_action → monitor_closely → recommend_conmed → recommend_early_visit → recommend_hospital_visit`

We score against ALL ground-truth AEs (visual + audio + non-visual):
- **Recall**: fraction of true AEs detected
- **Precision**: fraction of detected AEs that are real
- **F1**: harmonic mean
- **Grade MAE**: mean absolute error in CTCAE grade estimation

In [ ]:
from src.experiments.eval_prompt_templates import generate_final_assessment, score_t4_assessment

assessment_results = {}
for r in results:
    resp = r["_full_response"]
    resp["_turn"] = 2
    resp["_role"] = "nurse"
    conversation = [{**t1_visible, "_role": "patient", "_turn": 1}, resp]

    assessment = generate_final_assessment(scenario, conversation, nurse_fn)
    score = score_t4_assessment(assessment, all_gt_aes)
    assessment_results[r["strategy"]] = {"assessment": assessment, "score": score}

    detected = [d.get('ae_term', '') for d in assessment.get('detected_aes', []) if isinstance(d, dict)]
    s = score
    print(
        f"  {r['strategy']:18s}  Recall={s['ae_recall']:.2f}  Precision={s['ae_precision']:.2f}"
        f"  F1={s['ae_f1']:.2f}  GradeMAE={s['grade_mae']:.1f}  → {assessment.get('action', '?')}"
        f"  detected={detected}"
    )

In [ ]:
gt_labels = ["{} (Grade {})".format(ae["ae_term"], ae["grade"]) for ae in all_gt_aes]
print(f"Ground Truth AEs: {gt_labels}\n")

top_strategies = sorted(
    assessment_results.items(),
    key=lambda x: x[1]["score"]["ae_f1"],
    reverse=True,
)[:3]

for strategy, data in top_strategies:
    a, s = data["assessment"], data["score"]
    print("=" * 60)
    print(f"  {strategy}")
    print("=" * 60)
    print(f"  Recommended action: {a.get('action', '?')}")
    print(f"  Reason: {str(a.get('action_reason', ''))[:150]}")
    print(f"  Concern level: {a.get('overall_concern_level', '?')}")
    print(f"  Detected AEs:")
    for d in a.get("detected_aes", []):
        if isinstance(d, dict):
            print(f"    • {d.get('ae_term', '?')} — Grade {d.get('estimated_grade', '?')} (confidence: {d.get('confidence', '?')})")
            print(f"      Evidence: {str(d.get('evidence', ''))[:120]}")
    print(f"  Score: Recall={s['ae_recall']:.2f}  Precision={s['ae_precision']:.2f}  F1={s['ae_f1']:.2f}  GradeMAE={s['grade_mae']:.1f}")
    print()

---

## Part 6 — Multimodal Ablation Study

How much does each modality contribute to AE detection?
We run the **Deployment-Ready** strategy with different input combinations:

| Condition | Camera | Mic | Conversation |
|-----------|--------|-----|-------------|
| Full Multimodal | ✓ | ✓ | ✓ |
| Visual + Text | ✓ | ✗ | ✓ |
| Audio + Text | ✗ | ✓ | ✓ |
| Text Only | ✗ | ✗ | ✓ |

In [ ]:
EMPTY_VIS = {"source": "N/A", "findings": [], "general_observations": []}

ablation_configs = {
    "Full Multimodal (V+A+T)": {"visual": visual_assessment, "audio": audio_assessment},
    "Visual + Text (V+T)":     {"visual": visual_assessment, "audio": ""},
    "Audio + Text (A+T)":      {"visual": EMPTY_VIS, "audio": audio_assessment},
    "Text Only (T)":           {"visual": EMPTY_VIS, "audio": ""},
}

print("Running ablation with Deployment-Ready strategy ...\n")

ablation_results = {}
for config_name, config in ablation_configs.items():
    s_mod = {**scenario, "visual_assessment": config["visual"], "audio_assessment": config["audio"]}

    sys_prompt, usr_prompt = template_f_realistic(
        s_mod, quality, turn=2, history=[{**t1_visible, "_role": "patient", "_turn": 1}]
    )
    t2 = nurse_fn(sys_prompt, usr_prompt)
    t2["_turn"] = 2
    t2["_role"] = "nurse"

    conversation = [{**t1_visible, "_role": "patient", "_turn": 1}, t2]
    assessment = generate_final_assessment(s_mod, conversation, nurse_fn)
    score = score_t4_assessment(assessment, all_gt_aes)

    ablation_results[config_name] = {"assessment": assessment, "score": score}
    detected = [d.get("ae_term", "") for d in assessment.get("detected_aes", []) if isinstance(d, dict)]
    print(f"  {config_name:30s}  Recall={score['ae_recall']:.2f}  F1={score['ae_f1']:.2f}  detected={detected}")

# ── Comparison table ──
rows = []
for name, data in ablation_results.items():
    s = data["score"]
    rows.append({
        "Condition": name,
        "Recall": s["ae_recall"],
        "Precision": s["ae_precision"],
        "F1": s["ae_f1"],
        "Grade Error": s["grade_mae"],
        "# Detected": s["n_detected"],
    })

df_abl = pd.DataFrame(rows).set_index("Condition")
print()
display(HTML("<h3>Multimodal Ablation — Deployment-Ready Strategy</h3>"))
display(
    df_abl.style.format("{:.3f}", subset=["Recall", "Precision", "F1", "Grade Error"])
    .format("{:.0f}", subset=["# Detected"])
    .highlight_max(axis=0, subset=["Recall", "F1"], color="#c6efce")
    .highlight_min(axis=0, subset=["Grade Error"], color="#c6efce")
)

---

## Part 7 — Aggregated Evaluation (8 Patients × 3 Rounds)

The live demo above shows one patient. Below we load **pre-computed** results across
**8 out-of-distribution test patients** with 3 nurse conversation rounds each.

We evaluate two aspects:

**A. During conversation** — Does the nurse ask the right questions?
- **% AEs Probed**: fraction of ground-truth AEs targeted by nurse questions
- **Patient Comfort**: did the patient stay engaged? (0-1)
- **Overall**: combined score (high only when AEs detected WITHOUT alienating patient)

**B. Final assessment** — Is the conclusion correct?
- **Recall / Precision / F1**: standard classification metrics
- **Grade Error**: CTCAE grade estimation error (0 = perfect)

In [ ]:
with open(DATA_DIR / "care_ai_eval_data" / "eval_prompt_templates_baseline.json") as f:
    res = json.load(f)

INTERNAL_TO_DISPLAY = {
    "A_current": "Baseline", "B_concise": "Concise", "C_rich": "Full Context",
    "D_minimal": "No Drug Info", "E_fewshot": "Few-Shot", "F_realistic": "Deployment-Ready",
}

# ── Table A: Conversation quality ──
conv_rows = []
for internal, display_name in INTERNAL_TO_DISPLAY.items():
    r = res[internal]
    prog = r["turn_ae_progression"]
    conv_rows.append({
        "Prompt Strategy": display_name,
        "% AEs Probed": r["avg_ae"],
        "Patient Comfort": r["avg_mood"],
        "Overall": r["avg_pareto"],
        "After Round 1": prog[0], "After Round 2": prog[1], "After Round 3": prog[2],
    })
df_conv = pd.DataFrame(conv_rows).set_index("Prompt Strategy")

display(HTML("<h3>A. During Conversation — Does the nurse ask the right questions?</h3>"))
display(df_conv.style.format("{:.3f}")
    .highlight_max(axis=0, subset=["% AEs Probed", "Overall"], color="#c6efce"))

# ── Table B: Final assessment accuracy ──
assess_rows = []
for internal, display_name in INTERNAL_TO_DISPLAY.items():
    r = res[internal]
    assess_rows.append({
        "Prompt Strategy": display_name,
        "Recall": r["t4_recall"],
        "Precision": r["t4_precision"],
        "F1": r["t4_f1"],
        "Grade Error": r["t4_grade_mae"],
    })
df_assess = pd.DataFrame(assess_rows).set_index("Prompt Strategy")

display(HTML("<h3>B. Final Assessment — Is the nurse's conclusion correct?</h3>"))
display(df_assess.style.format("{:.3f}")
    .highlight_max(axis=0, subset=["Recall", "F1"], color="#c6efce")
    .highlight_min(axis=0, subset=["Grade Error"], color="#c6efce"))

In [ ]:
display(HTML("<h3>How detection improves with each round of conversation</h3>"))
display(HTML("<p><em>% AEs Probed = fraction of ground-truth AEs the nurse has asked about by that round</em></p>"))

prog_rows = []
for internal, display_name in INTERNAL_TO_DISPLAY.items():
    ae = res[internal]["turn_ae_progression"]
    mood_prog = res[internal]["turn_mood_progression"]
    prog_rows.append({
        "Strategy": display_name,
        "% AEs Probed (Round 1)": ae[0],
        "% AEs Probed (Round 2)": ae[1],
        "% AEs Probed (Round 3)": ae[2],
        "Patient Comfort (Round 1)": mood_prog[0],
        "Patient Comfort (Round 2)": mood_prog[1],
        "Patient Comfort (Round 3)": mood_prog[2],
    })

df_prog = pd.DataFrame(prog_rows).set_index("Strategy")
display(df_prog.style.format("{:.3f}").background_gradient(cmap="YlGn", axis=None))

---

## Conclusions

In [ ]:
templates = list(INTERNAL_TO_DISPLAY.keys())
best_pareto_k = max(templates, key=lambda t: res[t]["avg_pareto"])
best_t4_k = max(templates, key=lambda t: res[t]["t4_f1"])
best_ae_k = max(templates, key=lambda t: res[t]["avg_ae"])

bp = INTERNAL_TO_DISPLAY[best_pareto_k]
bt = INTERNAL_TO_DISPLAY[best_t4_k]
ba = INTERNAL_TO_DISPLAY[best_ae_k]

findings = f"""
### Results

| What we measured | Best strategy | Score |
|---|---|---|
| Best at detecting AEs while keeping patient engaged | **{bp}** | Overall = {res[best_pareto_k]['avg_pareto']:.3f} |
| Most accurate final clinical assessment | **{bt}** | F1 = {res[best_t4_k]['t4_f1']:.3f} (Recall = {res[best_t4_k]['t4_recall']:.3f}) |
| Most AEs probed during conversation | **{ba}** | {res[best_ae_k]['avg_ae']:.1%} of ground-truth AEs asked about |

### Key Takeaways

1. **Deployment-Ready prompt is the best overall strategy** — it achieves the highest combined AE detection + patient comfort, using only information realistically available in production (drug profile, visual assessment, audio assessment, speaking style guidelines).

2. **Multimodal input improves AE detection** — The Deployment-Ready strategy uniquely receives audio assessment from HeAR, enabling it to flag cough-related AEs (pneumonitis) that text-only strategies miss.

3. **More rounds = more AEs found** — Deployment-Ready detects {res['F_realistic']['turn_ae_progression'][0]:.0%} of AEs after Round 1 → {res['F_realistic']['turn_ae_progression'][2]:.0%} after Round 3, a {res['F_realistic']['turn_ae_progression'][2]/max(res['F_realistic']['turn_ae_progression'][0],0.01):.1f}× improvement through multi-round probing.

4. **Patient personality data is not needed** — Full Context (which includes unrealistic personality data) does not outperform Deployment-Ready, suggesting MedGemma can adapt its approach from conversational cues alone.

5. **Strong intrinsic medical knowledge** — Even No Drug Info (no AE profile provided) achieves competitive detection, indicating MedGemma's pre-training covers drug-specific adverse event knowledge.

### End-to-End Pipeline Validated

This notebook demonstrates that the full multimodal pipeline works:
```
Patient Camera → MedGemma-Vision → visual_assessment ─┐
                                                       ├→ MedGemma-4B Nurse → Clinical Assessment
Patient Mic    → HeAR            → audio_assessment ──┘
```
Real model outputs (not mock data) flow from NB2's detection models into the nurse conversation,
demonstrating a practical telehealth monitoring system for clinical trials.
"""
display(Markdown(findings))

In [ ]:
del nurse_model, tokenizer
torch.cuda.empty_cache()
print("All models unloaded. Notebook complete.")